# 📊 Sentiment Analysis Pipeline
### Text Preprocessing · HTML Cleaning · VADER Sentiment Scoring · Visualizations

This notebook:
1. Accepts any `.txt` file (may contain raw HTML)
2. Strips HTML tags & attributes
3. Preprocesses text (tokenization, stopword removal, lemmatization)
4. Runs **VADER** sentiment analysis at document and sentence level
5. Produces sentence-level breakdowns, word clouds, and rich charts

---
## 0. Install & Import Dependencies

In [ ]:
#!pip install nltk vaderSentiment beautifulsoup4 matplotlib seaborn wordcloud pandas numpy -q

In [ ]:
import re, os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup
from wordcloud import WordCloud

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

for resource in ['punkt', 'stopwords', 'wordnet', 'punkt_tab']:
    nltk.download(resource, quiet=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
print('✅ Imports complete')

---
## 1. Load Input File

> **Change `FILE_PATH` to point to your `.txt` file.**  
> Use a raw string `r'...'` for Windows paths to avoid unicode errors.

In [ ]:
# ─────────────────────────────────────────────────────────────
FILE_PATH = r'C:\Users\dell\Downloads\Infosys internship\final\nlp_preprocessing_2.txt'   # ← CHANGE THIS
# ─────────────────────────────────────────────────────────────

with open(FILE_PATH, 'r', encoding='utf-8', errors='replace') as f:
    raw_text = f.read()

print(f'✅ Loaded "{FILE_PATH}" — {len(raw_text):,} characters, {raw_text.count(chr(10)):,} lines')
print('\n--- Raw Text Preview (first 500 chars) ---')
print(raw_text[:500])

---
## 2. Preprocessing Pipeline

### 2a. HTML Tag & Attribute Removal

In [ ]:
def remove_html(text):
    """Remove all HTML tags, attributes, scripts and styles."""
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style', 'head', 'meta', 'link']):
        tag.decompose()
    clean = soup.get_text(separator=' ')
    return re.sub(r'\s+', ' ', clean).strip()

html_cleaned = remove_html(raw_text)
print('=== After HTML Removal ===')
print(html_cleaned[:500])
print(f'\nLength: {len(raw_text):,} → {len(html_cleaned):,} chars (removed {len(raw_text)-len(html_cleaned):,})')

### 2b. Text Normalization & Deep Clean

In [ ]:
def normalize_text(text):
    """Lowercase, remove URLs, emails, special characters."""
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

normalized = normalize_text(html_cleaned)
print('=== After Normalization ===')
print(normalized[:400])

### 2c. Tokenization, Stopword Removal & Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()
STOP_WORDS  = set(stopwords.words('english'))

# Sentence tokenize on html_cleaned (preserves punctuation for VADER)
sentences_raw = sent_tokenize(html_cleaned)

# Full token pipeline on normalized text
all_tokens = word_tokenize(normalized)
filtered_tokens = [
    lemmatizer.lemmatize(tok)
    for tok in all_tokens
    if tok.isalpha() and tok not in STOP_WORDS and len(tok) > 2
]

print(f'Total sentences  : {len(sentences_raw)}')
print(f'Total tokens     : {len(all_tokens):,}')
print(f'After filtering  : {len(filtered_tokens):,}')
print(f'Unique lemmas    : {len(set(filtered_tokens)):,}')
print(f'\nSample filtered tokens: {filtered_tokens[:20]}')

---
## 3. Preprocessing Summary Chart

In [ ]:
summary = pd.DataFrame({
    'Stage': ['Raw Input', 'After HTML Removal', 'After Normalization',
              'Tokenized', 'After Stopword + Lemma Filter'],
    'Count': [len(raw_text), len(html_cleaned), len(normalized),
              len(all_tokens), len(filtered_tokens)]
})

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']
bars = ax.barh(summary['Stage'], summary['Count'], color=colors)
ax.bar_label(bars, fmt='{:,.0f}', padding=4)
ax.set_xlabel('Count (chars for first 3, tokens for last 2)')
ax.set_title('Text Reduction Through Preprocessing Pipeline', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. VADER Sentiment Analysis

### 4a. Document-Level Score

In [ ]:
vader = SentimentIntensityAnalyzer()
v_scores = vader.polarity_scores(html_cleaned)

def vader_label(c):
    return 'POSITIVE' if c >= 0.05 else ('NEGATIVE' if c <= -0.05 else 'NEUTRAL')

print('=' * 50)
print('     DOCUMENT-LEVEL SENTIMENT REPORT')
print('=' * 50)
print(f'  Overall Sentiment  : {vader_label(v_scores["compound"])}')
print(f'  Compound Score     : {v_scores["compound"]:+.4f}')
print(f'  Positive words     : {v_scores["pos"]*100:.1f}%')
print(f'  Neutral  words     : {v_scores["neu"]*100:.1f}%')
print(f'  Negative words     : {v_scores["neg"]*100:.1f}%')
print('=' * 50)
if v_scores['neu'] > 0.70:
    print()
    print('  ⚠️  NOTE: Neutral% is very high — this appears to be a')
    print('  technical/formal document. VADER compound may be inflated')
    print('  because it only sums the sentiment-bearing words and')
    print('  ignores the large neutral portion entirely.')

### 4b. Sentence-Level Analysis

In [ ]:
rows = []
for i, sent in enumerate(sentences_raw, 1):
    if len(sent.strip()) < 5:
        continue
    vs = vader.polarity_scores(sent)
    rows.append({
        'Sentence #': i,
        'Text':       sent.strip(),
        'Compound':   round(vs['compound'], 4),
        'Positive':   round(vs['pos'], 4),
        'Neutral':    round(vs['neu'], 4),
        'Negative':   round(vs['neg'], 4),
        'Label':      vader_label(vs['compound'])
    })

df = pd.DataFrame(rows)

def color_label(val):
    palette = {
        'POSITIVE': 'background-color:#d4edda;color:#155724',
        'NEGATIVE': 'background-color:#f8d7da;color:#721c24',
        'NEUTRAL':  'background-color:#fff3cd;color:#856404'
    }
    return palette.get(val, '')

display(
    df[['Sentence #', 'Text', 'Compound', 'Positive', 'Neutral', 'Negative', 'Label']]
    .style
    .applymap(color_label, subset=['Label'])
    .format({'Compound': '{:+.4f}', 'Positive': '{:.4f}',
             'Neutral': '{:.4f}', 'Negative': '{:.4f}'})
    .set_caption('Sentence-Level VADER Scores')
)

---
## 5. Visualizations

### 5a. Sentiment Score Gauge + Distribution Pie

In [ ]:
COLORS = {'POSITIVE': '#2ecc71', 'NEUTRAL': '#f1c40f', 'NEGATIVE': '#e74c3c'}
compound = v_scores['compound']
color    = '#2ecc71' if compound >= 0.05 else '#e74c3c' if compound <= -0.05 else '#f1c40f'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gauge bar
ax1 = axes[0]
ax1.barh([''], [2], left=[-1], color='#ecf0f1', height=0.45)
ax1.barh([''], [compound], color=color, height=0.45, alpha=0.9)
ax1.axvline(0, color='#2c3e50', linewidth=1.5, linestyle='--')
ax1.set_xlim(-1, 1)
ax1.set_yticks([])
offset = 0.04 if compound >= 0 else -0.04
ha     = 'left' if compound >= 0 else 'right'
ax1.text(compound + offset, 0, f'{compound:+.3f}', va='center', ha=ha,
         fontweight='bold', color='#2c3e50', fontsize=14)
ax1.set_title(f'Overall Sentiment: {vader_label(compound)}\n(VADER Compound Score)',
              fontweight='bold', fontsize=13)
ax1.set_xlabel('← Negative                       Positive →', fontsize=10)

# Sentence distribution pie
ax2 = axes[1]
counts = df['Label'].value_counts()
pie_colors = [COLORS.get(k, '#95a5a6') for k in counts.index]
wedges, texts, autotexts = ax2.pie(
    counts, labels=counts.index, autopct='%1.1f%%',
    colors=pie_colors, startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight('bold')
ax2.set_title('Sentence-Level\nSentiment Distribution', fontweight='bold', fontsize=13)

plt.suptitle('VADER Sentiment Overview', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 5b. Sentiment Flow Across sentences

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

x        = df['Sentence #']
compound = df['Compound']

ax.fill_between(x, compound, where=(compound >= 0), interpolate=True,
                color='#2ecc71', alpha=0.35, label='Positive region')
ax.fill_between(x, compound, where=(compound < 0), interpolate=True,
                color='#e74c3c', alpha=0.35, label='Negative region')
ax.plot(x, compound, 'k-o', markersize=5, linewidth=1.8, label='Compound score')
ax.axhline(0,     color='gray',    linestyle='--', linewidth=1)
ax.axhline( 0.05, color='#27ae60', linestyle=':',  linewidth=1, alpha=0.7)
ax.axhline(-0.05, color='#c0392b', linestyle=':',  linewidth=1, alpha=0.7)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel('Sentence Number', fontsize=11)
ax.set_ylabel('VADER Compound Score', fontsize=11)
ax.set_title('Sentiment Flow Across Sentences', fontweight='bold', fontsize=13)
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

### 5c. Stacked Pos / Neu / Neg Bar per Sentence

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
x     = np.arange(len(df))
width = 0.6

ax.bar(x, df['Positive'], width, label='Positive', color='#2ecc71')
ax.bar(x, df['Neutral'],  width, bottom=df['Positive'], label='Neutral', color='#f1c40f')
ax.bar(x, df['Negative'], width, bottom=df['Positive'] + df['Neutral'],
       label='Negative', color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels([f"S{int(n)}" for n in df['Sentence #']], rotation=45, ha='right')
ax.set_ylabel('Proportion')
ax.set_ylim(0, 1)
ax.set_title('VADER Positive / Neutral / Negative Breakdown per Sentence',
             fontweight='bold', fontsize=13)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 5d. Top Word Frequency

In [ ]:
freq    = FreqDist(filtered_tokens)
top_n   = 20
top_words = freq.most_common(top_n)

if top_words:
    words, counts = zip(*top_words)
    fig, ax = plt.subplots(figsize=(12, 5))
    palette = sns.color_palette('husl', len(words))
    bars = ax.bar(words, counts, color=palette, edgecolor='white', linewidth=1)
    ax.bar_label(bars, fontsize=9)
    ax.set_xlabel('Word')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Top {top_n} Most Frequent Words (after stopword removal)',
                 fontweight='bold')
    plt.xticks(rotation=40, ha='right')
    plt.tight_layout()
    plt.show()

### 5e. Word Clouds — Overall · Positive · Negative

In [ ]:
def collect_words(sentences_df, label_val):
    texts  = sentences_df[sentences_df['Label'] == label_val]['Text'].tolist()
    tokens = []
    for t in texts:
        toks = word_tokenize(normalize_text(t))
        tokens += [lemmatizer.lemmatize(tok)
                   for tok in toks
                   if tok.isalpha() and tok not in STOP_WORDS and len(tok) > 2]
    return ' '.join(tokens)

overall_text  = ' '.join(filtered_tokens)
positive_text = collect_words(df, 'POSITIVE')
negative_text = collect_words(df, 'NEGATIVE')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
configs = [
    (overall_text,  'All Words',      'viridis'),
    (positive_text, 'Positive Words', 'Greens'),
    (negative_text, 'Negative Words', 'Reds'),
]

for ax, (txt, title, cmap) in zip(axes, configs):
    if txt.strip():
        wc = WordCloud(width=500, height=300, background_color='white',
                       colormap=cmap, max_words=80, collocations=False).generate(txt)
        ax.imshow(wc, interpolation='bilinear')
    else:
        ax.text(0.5, 0.5, 'No sentences in\nthis category',
                ha='center', va='center', transform=ax.transAxes, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle('Word Clouds', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Final Summary Report

In [ ]:
total_sents = len(df)
v_counts    = df['Label'].value_counts()

print('=' * 54)
print('         FINAL SENTIMENT ANALYSIS REPORT')
print('=' * 54)
print(f'  File analysed     : {FILE_PATH}')
print(f'  Total sentences   : {total_sents}')
print(f'  Total tokens      : {len(all_tokens):,}')
print(f'  Unique lemmas     : {len(set(filtered_tokens)):,}')
print()
print('  VADER (document-level)')
print(f'  Overall sentiment : {vader_label(v_scores["compound"])}')
print(f'  Compound score    : {v_scores["compound"]:+.4f}')
print(f'  Pos {v_scores["pos"]:.3f}  |  Neu {v_scores["neu"]:.3f}  |  Neg {v_scores["neg"]:.3f}')
print()
print('  Sentence Breakdown')
for lbl in ['POSITIVE', 'NEUTRAL', 'NEGATIVE']:
    cnt = v_counts.get(lbl, 0)
    pct = cnt / total_sents * 100 if total_sents else 0
    bar = '█' * int(pct / 5)
    print(f'  {lbl:<10}: {cnt:>3} ({pct:5.1f}%)  {bar}')
print()
print('  Top 10 Keywords')
for w, c in freq.most_common(10):
    print(f'  {w:<22} {c}')
print('=' * 54)

---
## 7. Export Results to CSV

In [ ]:
output_csv = 'sentiment_results.csv'
df.to_csv(output_csv, index=False)
print(f'✅ Results exported to "{output_csv}"')
display(df.head())